# 19.20 Causal inference (do-calculus, counterfactuals, propensity)

Causal inference separates seeing an association from estimating what an intervention would change.

The notebook builds treatment/outcome data with known effects and rising confounding. IPW estimates the intervention effect, while a collider pitfall shows why the adjustment set must be declared before looking at outcomes.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1919
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _standardize(X):
    scaler = StandardScaler()
    return scaler.fit_transform(np.asarray(X, dtype=float))


def _binary_target(y):
    y = np.asarray(y)
    classes = np.unique(y)
    if len(classes) == 2:
        return (y == classes[-1]).astype(int)
    return (y == classes[-1]).astype(int)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def fit_logistic(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)
    return clf


## Build IPW once on D1\n\nThe lesson formula is $$ATE=\mathbb{E}\left[\frac{T Y}{e(X)}-\frac{(1-T)Y}{1-e(X)}\right].$$\nThe lesson components satisfy $1.200+0.800+0.400=2.400$ and the leading share is $1.200/2.400=0.500$.

In [ ]:

def estimate_ate_ipw(T, Y, e, clip=0.05):
    T = np.asarray(T, dtype=float)
    Y = np.asarray(Y, dtype=float)
    e = np.clip(np.asarray(e, dtype=float), clip, 1.0 - clip)
    treated = T * Y / e
    control = (1.0 - T) * Y / (1.0 - e)
    return float(np.mean(treated - control))


def allowed_adjustment_set(adjust_for, colliders):
    adjust_for = set(adjust_for)
    colliders = set(colliders)
    return len(adjust_for.intersection(colliders)) == 0

components = np.array([1.200, 0.800, 0.400])
total = float(components.sum())
share = float(abs(components[0]) / np.abs(components).sum())
toy_ate = estimate_ate_ipw(np.array([1, 0]), np.array([2.4, 0.0]), np.array([0.5, 0.5]))
assert round(total, 3) == 2.400
assert round(share, 3) == 0.500
assert round(toy_ate, 3) == 2.400
assert allowed_adjustment_set(["age", "severity"], ["post_treatment_score"])
print("toy IPW ATE", round(toy_ate, 3))
print("leading component share", round(share, 3))


Fit propensities from pre-treatment covariates only, clip overlap, and compare to the known simulated ATE.

In [ ]:

def fit_propensity(X, T):
    model = LogisticRegression(max_iter=2000)
    model.fit(X, T)
    return model.predict_proba(X)[:, 1]


def make_causal_dataset(X, y, confounding, effect, seed):
    local_rng = np.random.default_rng(seed)
    Xs = _standardize(X)
    yb = _binary_target(y)
    z = Xs[:, 0]
    logits = confounding * z - 0.2
    e_true = np.clip(sigmoid(logits), 0.03, 0.97)
    T = local_rng.binomial(1, e_true)
    if T.sum() == 0 or T.sum() == len(T):
        T = np.arange(len(T)) % 2
    baseline = 0.6 * z + 0.4 * yb + local_rng.normal(0.0, 0.25, len(yb))
    Y = baseline + effect * T
    collider = 0.7 * Y + 0.7 * T + local_rng.normal(0.0, 0.1, len(yb))
    observed = np.column_stack([Xs[:, : min(5, Xs.shape[1])], collider])
    return {
        "X_pre": Xs[:, : min(5, Xs.shape[1])],
        "X_with_collider": observed,
        "T": T,
        "Y": Y,
        "e_true": e_true,
        "true_ate": effect,
    }

first = make_causal_dataset(clf_ladder()[0][1], clf_ladder()[0][2], 0.2, 0.8, SEED)
e_hat = fit_propensity(first["X_pre"], first["T"])
ate_hat = estimate_ate_ipw(first["T"], first["Y"], e_hat)
print("known ATE", first["true_ate"])
print("estimated ATE", round(ate_hat, 3))
print("overlap", round(float(e_hat.min()), 3), round(float(e_hat.max()), 3))


## Dataset ladder

Each rung wraps the F15 classifier ladder in a treatment/outcome simulator with stronger confounding and weaker overlap.

In [ ]:

causal_specs = [
    ("D1 linear toy", 0.2, 0.8),
    ("D2 nonlinear confounding", 0.7, 0.8),
    ("D3 real tabular confounding", 1.2, 0.8),
    ("D4 image-derived confounding", 1.7, 0.8),
    ("D5 biased low-overlap policy", 2.4, 0.8),
]
causal_rungs = []
for idx, ((base_name, X, y), (name, confounding, effect)) in enumerate(zip(clf_ladder(), causal_specs)):
    data = make_causal_dataset(X, y, confounding, effect, SEED + idx)
    causal_rungs.append((name, data, confounding))
    print(name, data["X_pre"].shape, "treated", int(data["T"].sum()), "control", int((1 - data["T"]).sum()))
    print("sample T,Y,e", data["T"][:3], np.round(data["Y"][:3], 3), np.round(data["e_true"][:3], 3))


## Run IPW across D1-D5

In [ ]:

causal_results = []
for name, data, confounding in causal_rungs:
    e_hat = fit_propensity(data["X_pre"], data["T"])
    ate = estimate_ate_ipw(data["T"], data["Y"], e_hat)
    error = abs(ate - data["true_ate"])
    overlap_width = float(np.percentile(e_hat, 95) - np.percentile(e_hat, 5))
    causal_results.append((name, ate, data["true_ate"], error, overlap_width, e_hat))

print("rung | ate_hat | true_ate | abs_error | overlap_width")
for row in causal_results:
    print(row[0], *(round(float(v), 3) for v in row[1:5]))


## Results visualization

Left: treated/control propensity balance by rung. Right: ATE error as overlap stress rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for ax, (name, data, confounding) in zip(axes[:5], causal_rungs):
    e_hat = causal_results[[row[0] for row in causal_results].index(name)][5]
    ax.hist(e_hat[data["T"] == 1], bins=12, alpha=0.6, label="treated")
    ax.hist(e_hat[data["T"] == 0], bins=12, alpha=0.6, label="control")
    ax.set_title(name)
    ax.set_xlim(0, 1)
axes[0].legend()
axes[5].plot(range(1, 6), [row[3] for row in causal_results], marker="o")
axes[5].set_xticks(range(1, 6))
axes[5].set_xlabel("rung")
axes[5].set_ylabel("ATE absolute error")
axes[5].set_title("ATE error vs stress")
plt.tight_layout()


## Pitfall on D5: conditioning on a collider

The wrong estimator lets a post-treatment collider enter the propensity model. The fix declares pre-treatment covariates, clips propensities, and checks overlap.

In [ ]:

name, data, confounding = causal_rungs[-1]
e_bad = fit_propensity(data["X_with_collider"], data["T"])
e_fix = fit_propensity(data["X_pre"], data["T"])
bad_ate = estimate_ate_ipw(data["T"], data["Y"], e_bad)
fixed_ate = estimate_ate_ipw(data["T"], data["Y"], e_fix)
print("bad collider-adjusted ATE", round(bad_ate, 3))
print("fixed declared-set ATE", round(fixed_ate, 3))
print("known ATE", data["true_ate"])
print("clipped overlap", round(float(np.clip(e_fix, 0.05, 0.95).min()), 3), round(float(np.clip(e_fix, 0.05, 0.95).max()), 3))


## Evaluate it + Practice

- Metric: absolute ATE error against the known effect
- No-skill baseline: unadjusted treated-control mean difference
- Cheap sanity check: randomized D1 should have broad overlap
- Ablation: include the collider and the ATE should become less trustworthy
- Failure signals: propensities near 0 or 1, high weight concentration, or adjustment sets containing descendants

Practice prompts:
1. Change the D5 stress level and predict which metric should move first.

2. Add one subgroup or environment slice and explain whether the conclusion changes.

3. Replace the default logistic model with another CPU-only sklearn classifier and compare the same metric.